Пакеты, которые используются в этом блокноте:

# Глава 3: Программирование механизмов внимания

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0


<img src="https://camo.githubusercontent.com/47358e1fe19859b3a0c9e82bec664f8cbb5650da9b632a7a70fb4c2b6d63ca2c/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30312e776562703f313233" width="800px">

<img src="https://camo.githubusercontent.com/12846bf6a7a8cc9f40e16baf0d91d194c8fd3568dc2601f2371a4d514b81898e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30322e77656270" width="800px">

## 3.1 Проблема моделирования длинных последовательностей

- Перевод текста дословно невозможен из-за различий в грамматических структурах между исходным и целевым языками:

<img src="https://camo.githubusercontent.com/38734035d68e30c6d6f904352feae79b4f60859bda22d838fc29d5b0a6016703/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30332e77656270" width="800px">

- До появления трансформеров рекурентные нейронные сети (recurrent neural networks, RNN) были самой популярной архитектурой кодировщик-декодировщик для языкового перевода
- При такой настройке кодировщик обрабатывает последовательность токенов из исходного языка, используя скрытое состояние — своего рода промежуточный уровень в нейронной сети — для генерации сжатого представления всей входной последовательности:

<img src="https://camo.githubusercontent.com/65735d9325818c300bec445e93a922c66e3a64760f7ad4a0fedb3cde7991f039/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30342e77656270" width="800px">

 ---

**Скрытое состояние** — это **память** или **блокнот** нейронной сети прямо во время чтения предложения.

Вы читаете первое слово «Кот». Ваш блокнот: «Пока речь о коте».

Вы читаете второе слово «сидит». Блокнот обновляется: «Кто-то (кот) совершает действие (сидит)».

Вы читаете третье слово «на». Блокнот: «Кот сидит где-то (пока не знаем где)».

Вы читаете четвертое слово «столе». Блокнот финальный: «Кот сидит на столе».

**Это финальное содержимое блокнота как раз и есть «скрытое состояние» после обработки всего предложения.** Оно хранит в себе *сжатый смысл* всей фразы.

### Зачем это нужно?

Когда сеть-декодировщик (переводчик) начнет рождать английские слова «The cat sits on the...», она не будет перечитывать исходное предложение заново. Вместо этого она просто **посмотрит в ваш блокнот (скрытое состояние)** и поймет: «Ага, субъект — кот, действие — сидеть, место — стол».

### Как это реализовано технически (немного глубже)

Технически «скрытое состояние» — это **просто список чисел** (вектор), например, из 256, 512 или 1024 чисел.

С каждым новым словом происходит три шага:

1.  **Берем старый блокнот** (предыдущее скрытое состояние).
2.  **Берем новое слово** (превращенное в числа — вектор).
3.  **Нейронная сеть (ячейка RNN) выполняет очень простую формулу**:
    `Новый блокнот = функция(Старый блокнот, Новое слово)`

Эта функция — всегда одно и то же уравнение с весами (настройками, которые сеть выучила на миллионах примеров). Она решает: «Как сильно новое слово должно *изменить* старую память?»

**Важная деталь:** Числа внутри «блокнота» — это **не** конкретные слова («кот», «стол»). Это абстрактные признаки, которые сеть придумала сама:
- Первое число может означать «насколько действие уже завершено».
- Второе — «одушевленность субъекта».
- Третье — «активность глагола».
- ...и так далее. Человек эти числа не интерпретирует.

### Проблема RNN

Представьте, что вы читаете длинное предложение:
> «Тот большой рыжий кот, который сломал вчера вазу и которого мы выгнали, ... **сидел** на столе».

К тому моменту, как вы дочитали до слова «сидел», ваш **блокнот уже переполнен** информацией о «сломал», «вазу», «выгнали». А слово «кот» было в самом начале. Старое скрытое состояние уже много раз перезаписалось новыми словами.

Это называется **проблема забывания дальних связей**. RNN очень легко помнит последние 5-7 слов, но «кот» и «сидел» могут «разорваться» в памяти сети. Из-за этого трансформеры (с их вниманием) и пришли на смену RNN — они умеют смотреть прямо на любое слово из прошлого, не полагаясь на «один блокнот на все».

**Скрытое состояние** — это постоянно обновляющаяся «записка памяти», чтобы к концу предложения нейросеть помнила главное о начале.

---

## 3.2 Управление зависимостей данных с помощью механизмов привлечения внимания

- С помощью механизма внимания декодирующая часть сети, генерирующая текст, может выборочно обращаться ко всем входным токенам. Это означает, что некоторые входные токены более важны для генерации конкретного выходного токена, чем другие. Важность определяется весами внимания:

<img src="https://camo.githubusercontent.com/87f17c8ca381ec022193c5b6acd2c9a7d3cd8b000c632c848e92221c19a4ef00/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30352e77656270" width="800px">

- Самовнимание - механизм, который используется для вычисления более эффективных входных представлений. Он позволяет каждой позиции во входной последовательности взаимодействовать со всеми остальными позициями в той же последовательности и оценивать их вклад (важность) при вычислении представления последовательности

<img src="https://camo.githubusercontent.com/8c1e96b812fd6f79afb05d90a83b57e8c22ae6c8d98ba38420e933ebd20606fa/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30362e77656270" width="800px">

## 3.3 Обращение к разным частям входных данных с помощью самовнимания

### 3.3.1 Простой механизм самовнимания без обучаемых весов

- Предположим, что нам дана входная последовательность от $x ^ {(1)}$ до $x ^ {(T)}$
    - Входные данные представляют собой текст (например, предложение типа "Your journey starts with one step"), который уже преобразован во встроенные токены.
    - Например, $x ^{(1)}$ - это d-мерный вектор, представляющий слово "Ваш", и так далее
- ** Цель: ** вычислить контекстные векторы $z ^{(i)}$ для каждого элемента входной последовательности $x^{(i)}$ от $x ^{(1)}$ до $x^{(T)}$ (где $z$ и $x$ имеют тот же размер)
    - Вектор контекста $z^{(i)}$ представляет собой взвешенную сумму входных данных от $x^{(1)}$ до $x^{(T)}$
    - Вектор контекста является "контекстно" зависимым от определенных входных данных
    - Вместо $x^{(i)}$ в качестве заполнителя для произвольного входного токена давайте рассмотрим второй входной токен, $x^{(2)}$
    - И чтобы продолжить с конкретным примером, вместо заполнителя $z^{(i)}$ мы рассмотрим второй выходной вектор контекста, $z^{(2)}$
    - Второй контекстный вектор, $z ^{(2)}$, представляет собой взвешенную сумму по всем входным данным от $x ^{(1)}$ до $x^{(T)}$, взвешенную по отношению ко второму входному элементу $x^{(2)}$
    - Веса внимания - это веса, которые определяют, какой вклад вносит каждый из входных элементов во взвешенную сумму при вычислении $z ^ {(2)}.$
    - Короче говоря, представьте себе $z ^ {(2)}$ как модифицированную версию $x ^ {(2)}$, которая также включает в себя информацию обо всех других элементах ввода, имеющих отношение к данной задаче

<img src="https://camo.githubusercontent.com/a5f9c3d06c5fdc5345b311a9b0baf0e6601211a32b27822427d3275c4d4ec2d0/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30372e77656270" width="800px">

(Цифры на этом рисунке усечены до одной цифры после запятой, чтобы уменьшить визуальный беспорядок)

- По общему правилу ненормированные значения коэффициента внимания называются **"показателями внимания"**, тогда как нормализованные значения показателя внимания, сумма которых равна 1, называются **"весами внимания".**

- Приведенный ниже код шаг за шагом повторяет приведенный выше рисунок

</br>

- **Шаг 1:** вычислите ненормализованные показатели внимания $\omega$
- Предположим, используется второй входной токен в качестве запроса, то есть $q^{(2)} = x^{(2)}$, вычисляются ненормализованные показатели внимания с помощью точечных произведений:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- Выше, $\omega$ - это греческая буква "омега", используемая для обозначения ненормализованных показателей внимания, q - элемент матрицы вложений, x - входное значение
    - Индекс "21" в $\omega_{21}$ означает, что элемент входной последовательности 2 использовался в качестве запроса к элементу входной последовательности 1

- Пусть  есть следующее входное предложение, которое уже встроено в трехмерные векторы:

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- Правилаа машинного и глубокого обучения - обучающие примеры представлены в виде строк, а значения объектов - в виде столбцов; в случае тензора, показанного выше, каждая строка представляет слово, а каждый столбец - измерение для встраивания

- Основная цель - продемонстрировать, как вектор контекста $z^{(2)}$ вычисляется с использованием второй входной последовательности, $x^{(2)}$, в виде запроса

- На рисунке показан начальный этап этого процесса, который включает в себя вычисление показателей внимания ω между $x ^ {(2)}$
и всеми другими входными элементами с помощью операции точечного умножения

<img src="https://camo.githubusercontent.com/128687a9a383db05f7715d04f5779621f407e76b668e807da298dfbf42ab3a43/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30382e77656270" width="800px">

- Используется элемент входной последовательности 2, $x ^{(2)}$, в качестве примера для вычисления вектора контекста $z^{(2)}$; позже будет обощение для вычисления всех векторов контекста.
- Первым шагом является вычисление ненормализованных показателей внимания путем вычисления точечного произведения между запросом $x^{(2)}$ и всеми другими входными токенами:

In [3]:
query = inputs[1]  # 2-й входной токен - это запрос

attn_scores_2 = torch.empty(inputs.shape[0]) # создает пустой массив из 3-х ячеек (shape - кортеж из библиотеки PyTorch)
for i, x_i in enumerate(inputs):
    # torch.dot() - скалярное произведение
    attn_scores_2[i] = torch.dot(x_i, query) # точечное произведение (транспонировать здесь не нужно, так как это векторы размером 1 дюйм)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


 ---

### Почему именно скалярное произведение? Почему нельзя было просто сложить числа? Почему именно перемножить и сложить?


### 1. Интуитивный уровень: "Совпадение интересов"

Представь, что векторы `x` (другие слова) и `q` (запрос) — это два списка ответов на вопросы анкеты. Оценки могут быть **положительными** (нравится) или **отрицательными** (не нравится).

**Пример:**
*   **Запрос (Кот):** Хочу того, кто: [Мягкий: +1, Большой: -1, Игривый: +1].
*   **Слово (Диван):** Я такой: [Мягкий: +1, Большой: +1, Сонный: -1].

**Как работают разные операции:**

1.  **Обычная сумма:** Мы тупо складываем цифры. Это как считать общую сумму баллов, игнорируя знаки.
    *   Результат не покажет конфликт. Если один хочет +100, а другой дает -100, их сумма будет 0, что *кажется* похожим на случай, когда оба хотят 0. Это **плохо**.

2.  **Разность (расстояние):** Мы вычитаем векторы ($x - q$). Мы ищем различия. Чем меньше разница, тем ближе. Это работает, но это **мера различия**, а нам нужна мера **совпадения** и **созвучия**. К тому же у разности теряется идея "усиления" при больших совпадениях.

3.  **Скалярное произведение (Умножение + Сложение):**
    *   Если оба числа имеют **одинаковый знак** (оба любят мягкость: $+1 \times +1 = +1$), мы получаем плюс и прибавляем его. Сотрудничество!
    *   Если знаки **противоположны** (кот хочет маленького, а диван большой: $-1 \times +1 = -1$), мы получаем минус и **штрафуем** результат. Конфликт интересов!
    *   Если кому-то **все равно** (ноль), то умножение на ноль обнуляет этот признак. Мы его не учитываем.

**Вывод:** Скалярное произведение — это **единственная простая операция, которая умеет награждать за согласие и наказывать за противоречие**, суммируя эти "плюсики" и "минусики" в одну общую оценку совместимости.

### 2. Геометрический уровень: "Угол обзора"

В школьной математике есть формула скалярного произведения:

$$\mathbf{x} \cdot \mathbf{q} = \|\mathbf{x}\| \cdot \|\mathbf{q}\| \cdot \cos(\theta)$$

Давай переведем:
*   $\|\mathbf{x}\|$ — длина вектора "Слова". (Насколько слово "длинное").
*   $\|\mathbf{q}\|$ — длина вектора "Запроса".
*   $\cos(\theta)$ — косинус угла между ними.

**Что делают сумма/разность?**
Они зависят от того, куда направлены оси координат, и от длины векторов. Два длинных вектора, смотрящие в разные стороны, могут дать такую же сумму, как два коротких, смотрящие в одну. Это хаос.

**Что делает скалярное произведение?**
Оно напрямую зависит от **угла**.
*   Если векторы смотрят **в одну сторону** (угол $0^\circ$, $\cos = 1$): произведение **МАКСИМАЛЬНОЕ**. Слова — синонимы.
*   Если векторы **перпендикулярны** (угол $90^\circ$, $\cos = 0$): произведение равно **НУЛЮ**. Слова не связаны.
*   Если векторы смотрят в **противоположные стороны** (угол $180^\circ$, $\cos = -1$): произведение **МАКСИМАЛЬНО ОТРИЦАТЕЛЬНОЕ**. Слова — антонимы.

А механизму внимания как раз и нужно ловить направление смысла: "Кот" и "Мурлыка" должны смотреть в одну сторону, "Кот" и "Бетон" — в перпендикулярные, "Горячий" и "Холодный" — в противоположные.

### 3. Математический уровень: "Билинейная форма"

Если посмотреть на реальную формулу Attention с матрицами $W_q$ и $W_k$:

$$\text{Score} = x_i^T W_q^T W_k x_j$$

Это называется **билинейной формой**. Это самый простой способ заставить нейросеть "обучиться" тому, как именно сравнивать два вектора. 

Матрицы $W$ как бы поворачивают пространство так, чтобы в новом пространстве нужные нам слова (например, глагол и его наречие) стали смотреть в одну сторону. А скалярное произведение — это идеальный инструмент, чтобы в этом новом пространстве померить косинус угла.

### Резюме

Скалярное произведение взяли за основу, потому что это **операция измерения "созвучия" (сонаправленности) двух наборов чисел, которая учитывает знаки**.

*   **Сумма** — это как измерять богатство человека, складывая его доходы и долги, не замечая, что долги — это минус.
*   **Скалярное произведение** — это как считать реальный баланс: доходы прибавляем, а долги вычитаем, понимая, что в сумме у него может оказаться минус, и он совсем не похож на богача.

---

- Дополнительное примечание: точечное произведение - это, по сути, сокращение для поэлементного умножения двух векторов и суммирования полученных результатов:

In [4]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **Шаг 2:** нормализация ненормализованных показателей внимания ("омеги", $\omega$) таким образом, чтобы они в сумме равнялись 1
- Вот простой способ нормализовать ненормализованные показатели внимания и суммировать их до 1 (условное обозначение, полезное для интерпретации и важное для стабильности обучения).:

<img src="https://camo.githubusercontent.com/bcd4dbb967d31efe0abcb5fcf77b533b3452d52dadaffee16bb74c128107b5b7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30392e77656270" width="800px">

In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Набор чисел:", attn_scores_2)
print("Веса внимания:", attn_weights_2_tmp)
print("Сумма:", attn_weights_2_tmp.sum())

Набор чисел: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Веса внимания: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Сумма: tensor(1.0000)


 ---

### Почему делим тензор на сумму его элементов? 

### 1. Магия бродкастинга - "магия PyTorch" (как тензор делится на число?)

Когда ты в PyTorch пишешь:
```python
тензор = torch.tensor([7.4, 20.1, 2.7])
сумма = тензор.sum() # -> 30.2 (одно число)
результат = тензор / сумма
```

Python понимает это по-человечески, а не математически-строго:
> *"Ага, у меня есть список из трех чисел и одно число. Пользователь, наверное, хочет поделить **каждый элемент** списка на это число. Я умный, я сделаю это для каждого по очереди!"*

**Python незримо для тебя разворачивает это в цикл:**
```python
# То, что написано:
attn_weights = scores / scores.sum()

# То, что Python делает внутри (приблизительно):
сумма = scores.sum() # 30.2
результат = []
for x in scores:
    результат.append(x / сумма) # Делим каждый элемент отдельно
```

Это называется **поэлементная операция** (element-wise operation). Ты даешь команду одной строкой, а компьютер применяет деление к каждой клеточке тензора.

### 2. Почему нам нужно делить каждую ячейку?

Вернемся к смыслу. До деления у нас просто "сила сигнала" (экспоненты):
`[7.4, 20.1, 2.7]`

Они неудобные. Если мы скажем нейросети: *"Возьми 20.1 частей слова №2 и смешай с 7.4 частями слова №1"*, получится винегрет непонятной концентрации.

Мы хотим сказать: *"Возьми **66%** слова №2 и **24%** слова №1"*.

Чтобы получить эти проценты, мы должны **каждую** силу сигнала (каждый элемент) разделить на **общую** силу (сумму).

**Процесс идет по ячейкам:**
1. Ячейка 0: `7.4 / 30.2 = 0.245` (Стало процентом)
2. Ячейка 1: `20.1 / 30.2 = 0.666` (Стало процентом)
3. Ячейка 2: `2.7 / 30.2 = 0.089` (Стало процентом)

Теперь у нас новый тензор `[0.245, 0.666, 0.089]`, который является нормированной "смесью".

### 3. Что было бы, если бы мы не делили?

Если бы этого деления не было, Attention не работал бы как вероятностный механизм.

Допустим, мы не поделили и пошли дальше — смешивать векторы `v` (Values).
В коротком предложении сумма внимания могла бы быть 30, в длинном — 500.
Масштаб выхода слоя Attention зависел бы от длины предложения. Это вызвало бы **взрыв градиентов** (числа становились бы то гигантскими, то крошечными при переходе от слоя к слою), и нейросеть бы просто сломалась (перестала обучаться).

Деление на сумму **фиксирует бюджет внимания** — у каждого токена всегда есть ровно 1 единица внимания, которую он может потратить на все остальные слова (включая себя).

У тебя есть три кучки яблок:
- Куча 1: 7 яблок
- Куча 2: 20 яблок
- Куча 3: 3 яблока
Всего 30 яблок.

Команда `тензор / тензор.sum()` говорит:
"А теперь выброси яблоки и оставь вместо них **долю от общего урожая**".
- 7 / 30 = 0.23 (23% урожая)
- 20 / 30 = 0.67 (67% урожая)
- 3 / 30 = 0.10 (10% урожая)

---

- Рекомендуется использовать функцию softmax для нормализации, которая лучше справляется с экстремальными значениями и обладает более желательными свойствами градиента во время обучения.
- Вот базовая реализация функции softmax для масштабирования, которая также нормализует векторные элементы таким образом, что они суммируются до 1:

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0) # dim=0  # суммируем "вниз" (по вертикали), сворачивая строки в одну 
                                                  # dim=1  # суммируем "вправо" (по горизонтали), сворачивая столбцы в один

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Веса внимания:", attn_weights_2_naive)
print("Сумма:", attn_weights_2_naive.sum())

Веса внимания: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Сумма: tensor(1.)


 ---

### Что делает код выше?

Это **Softmax** — улучшенная версия превращения баллов в проценты. Он решает проблемы, которые есть у простого деления на сумму.

### Строка 1-2: Функция `softmax_naive`

```python
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
```

**Что здесь происходит по шагам:**

1. **`torch.exp(x)`** — берем **экспоненту** (число $e \approx 2.718$) и возводим её в степень каждого элемента вектора `x`.
   - Для числа 2: $e^2 \approx 7.39$
   - Для числа 0: $e^0 = 1$
   - Для числа -2: $e^{-2} \approx 0.14$

2. **`torch.exp(x).sum(dim=0)`** — считаем сумму всех экспонент.

3. **`torch.exp(x) / сумма`** — делим каждую экспоненту на общую сумму.

Отличный вопрос! Ты спрашиваешь про разницу между **простым делением на сумму** и **Softmax**, а также про выбор `dim=0` или `dim=1`.

Давай разложим по полочкам, **что, когда и зачем применять**.

### `dim=0` vs `dim=1`

Представь тензор как таблицу:

```python
tensor = [
    [a, b, c],   # строка 0
    [d, e, f]    # строка 1
]
# shape = (2 строки, 3 столбца)
```

- **`dim=0`** → иду **вниз по строкам** (сворачиваю вертикально)
- **`dim=1`** → иду **вправо по столбцам** (сворачиваю горизонтально)

### Когда `dim=0`?

Суммируем **каждый столбец отдельно**. Результат — одна строка.

**Применяй, когда:**
- Хочешь узнать сумму/среднее по каждому **признаку** (колонке)
- Нормализуешь **батч** данных (каждый столбец отдельно)
- В Attention: **почти никогда** для весов внимания

**Пример:**
```python
оценки = torch.tensor([
    [5, 4, 3],  # ученик 1: матем, физика, инглиш
    [2, 5, 4]   # ученик 2
])

# dim=0: средний балл по каждому ПРЕДМЕТУ
оценки.sum(dim=0)  # [7, 9, 7] — сумма баллов по предметам
```

### Когда `dim=1`?

Суммируем **каждую строку отдельно**. Результат — один столбец.

**Применяй, когда:**
- Хочешь узнать сумму/среднее для каждого **объекта** (строки)
- **В Attention — всегда `dim=1` или `dim=-1`** для весов внимания!
- Нормализуешь вероятности внутри одного запроса

**Пример:**
```python
оценки = torch.tensor([
    [5, 4, 3],  # ученик 1
    [2, 5, 4]   # ученик 2
])

# dim=1: сумма баллов каждого УЧЕНИКА
оценки.sum(dim=1)  # [12, 11] — общий балл каждого
```

В Attention всегда `dim=-1` (последняя ось)

```python
scores = torch.tensor([
    [0.5, 0.2, 0.8],   # запрос 1 → ключи
    [0.1, 0.9, 0.3]    # запрос 2 → ключи
])

# dim=1: каждый запрос получает свои 100% внимания
weights = torch.softmax(scores, dim=1)
# weights[0] = [0.35, 0.26, 0.39]  (сумма = 1)
# weights[1] = [0.25, 0.55, 0.20]  (сумма = 1)
```

`dim=-1` работает как `dim=1` для 2D, но универсальнее: работает для любого числа измерений (всегда берет последнюю ось).


| Ситуация | Инструмент | `dim` |
|-||-|
| Учебный пример, всё положительное | `x / sum(x)` | не важно |
| Реальная нейросеть, любые числа | `softmax(x)` | `dim=-1` |
| Один вектор (одномерный) | любой метод | `dim=0` |
| Матрица attention, нормируем запросы | `softmax(x, dim=-1)` | `dim=-1` |
| Нормируем батч или признаки | зависит от задачи | `dim=0` или `dim=1` |

### Строка 3: Применяем функцию

```python
attn_weights_2_naive = softmax_naive(attn_scores_2)
```

Берем наши сырые баллы (например, `[0.48, 0.74, 0.35]`) и пропускаем через softmax.

### Строка 5-6: Печатаем результат

```python
print("Веса внимания:", attn_weights_2_naive)
print("Сумма:", attn_weights_2_naive.sum())
```

Смотрим на проценты и проверяем, что сумма = 1.

### Почему именно `exp(x)`, а не просто `x / sum(x)`?

У простого деления на сумму есть **две проблемы**:

### Проблема 1: Отрицательные числа

Если в баллах есть отрицательное число:

```python
баллы = [5, 2, -3]
сумма = 5 + 2 + (-3) = 4

# Простое деление:
веса = [5/4, 2/4, -3/4] = [1.25, 0.5, -0.75]
```

💥 **Катастрофа!** У нас появился **отрицательный процент** (-0.75), что бессмысленно. Нельзя "уделить -75% внимания" слову.

**Softmax спасает:**
- $e^{-3} \approx 0.05$ (это положительное число, хоть и маленькое!)
- Отрицательные баллы получают мизерные, но **положительные** веса.

### Проблема 2: Неразличимость слабых сигналов

Представь, что у нас очень маленькие баллы:

```python
баллы = [0.5, 0.4, 0.1]
сумма = 1.0

# Простое деление:
веса = [0.5, 0.4, 0.1]  # почти не изменилось
```

Все веса очень близки друг к другу. Модель "не уверена", на кого смотреть — всё кажется одинаково важным.

**Softmax с `exp` усиливает контраст:**
- `e^0.5 ≈ 1.65` → вес `1.65 / 4.22 ≈ 0.39`
- `e^0.4 ≈ 1.49` → вес `1.49 / 4.22 ≈ 0.35`
- `e^0.1 ≈ 1.11` → вес `1.11 / 4.22 ≈ 0.26`

Разница стала выразительнее! `exp` "растягивает" пространство вокруг больших чисел.

### Проблема 3: Экстремально большие числа

Если приходит очень большой балл:

```python
баллы = [100, 2, 1]
# Простое деление: 100/103 ≈ 0.97, остальные почти 0
```

Всё и так работает, но `exp(100)` — это астрономически огромное число (больше, чем атомов во Вселенной). Компьютер может "захлебнуться".

**На практике** из баллов предварительно вычитают максимум (`x - max(x)`), чтобы избежать слишком больших чисел, но в учебном коде это опускают для простоты.

### Сравнение в таблице

| Метод | Формула | Работает с минусами? | Контрастность | Итог |
|-|||||
| Простое деление | $x / \sum x$ | ❌ Нет | Слабая | Только для учебных примеров |
| **Softmax** | $e^x / \sum e^x$ | ✅ Да | Сильная | **Используется везде** |

### Метафора с экзаменом

Представь, что баллы — это оценки за экзамен:

- **Простое деление:** Ученик получил 5, другой 3, третий -2. Делим на общую сумму — третьему достается **отрицательная доля** внимания учителя. Так не бывает!
- **Softmax с exp:** Учитель каждую оценку "усиливает" через экспоненту. Двойка становится крошечным положительным числом (учитель всё равно чуть-чуть посмотрит на двоечника), а пятёрка становится огромной и забирает почти всё внимание. При этом суммарное внимание = 100%.

Обычное деление на сумму — это как раздать всем поровну по куску пиццы, даже тому, кто сказал, что не голоден (и даже тому, кто сказал "я на диете" — ему отрицательный кусок?).

Softmax — это волшебная печка:
1. Она превращает любой ответ в положительное "да" (даже если человек сказал "нет", печка делает это маленьким "да-ам").
2. Она делает громкий голос (большое число) еще громче, а шепот — еще тише.
3. В конце все голоса в сумме дают ровно 1 голос внимания.

---

- Базовая реализация, описанная выше, может страдать от проблем с числовой нестабильностью при больших или малых входных значениях из-за проблем с переполнением и недостаточным расходом
- Следовательно, на практике рекомендуется использовать реализацию PyTorch в softmax, которая была оптимизирована для повышения производительности:

In [7]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Веса внимания:", attn_weights_2)
print("Сумма:", attn_weights_2.sum())

Веса внимания: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Сумма: tensor(1.)


- **Шаг 3**: вычисление вектора контекста $z^{(2)}$ путем умножения встроенных входных токенов, $x^{(i)}$ на значения внимания и суммирование полученных векторов:

<img src="https://camo.githubusercontent.com/477fbbc39086ac657d7f6e6bc176e7f9f27f7784e5e7673c71ca89a0c17c57c4/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31302e77656270" width="800px">

In [8]:
query = inputs[1] # 2-й входной токен - это запрос

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


 ---

### Зачем умножать встроенные входные токены на значения внимания и суммировать полученные вектора? Почему именно такая формула?

### 1. Почему нельзя просто "взять слово с самым большим весом" (Hard Attention)?

Допустим, у нас веса: Кот — 0.67, Пушистый — 0.24, Спит — 0.09.

Можно было бы тупо выбрать победителя: **Кот (0.67)** и сказать: «Окей, слово "пушистый" в этом предложении означает "кот"».

**Почему это плохо:**
1.  **Потеря нюансов.** Смысл "пушистого" — это не просто "кот". Это "котовость" (67%) + "пушистость как свойство" (24%) + "сонное состояние" (9%).
2.  **Нет градиента.** Нейросеть учится через плавные изменения. Если мы просто выкидываем 33% информации, мы обрываем связи. Нельзя понять, как чуть-чуть изменить ответ, если мы огрубили результат до одного слова.



### 2. Почему нельзя "просто сложить векторы" (как в мешке слов)?

Допустим, мы взяли и тупо смешали все слова в кучу поровну: `x_кот + x_пушистый + x_спит`.

**Почему это плохо:**
Представь, что ты готовишь борщ. "Просто сложить" — это кинуть в кастрюлю все овощи целиком, не чистя, и залить водой.
Результат — "средняя температура по больнице".

*   Слово "Пушистый" теряет свою роль **вопроса**. Оно впитает в себя одинаково и важного "Кота", и неважный предлог "на", и слово "коврик". Получится каша.

**Что делает наша формула:** `sum(w_i * x_i)`.
Мы не просто складываем векторы. Мы **взвешиваем** их.
*   Мы кладем в борщ 67% капусты и только 9% соли. Мы регулируем концентрацию каждого ингредиента, чтобы получился нужный вкус (контекст).



### 3. Почему именно Умножение и Сложение? (Геометрия смысла)

Вектор — это точка в пространстве смыслов.
*   Точка "Кот" (вектор `x_0`)
*   Точка "Спит" (вектор `x_2`)

Мы не хотим оказаться ни в точке "Кот", ни в точке "Спит". Мы хотим оказаться **где-то между ними**, но ближе к "Коту".

Формула `w_0 * x_0 + w_2 * x_2` — это математический способ **нарисовать отрезок** между точками "Кот" и "Спит" и поставить новую точку на этом отрезке.

*   Если `w_0 = 1`, а `w_2 = 0` — мы в точке "Кот".
*   Если `w_0 = 0.5`, а `w_2 = 0.5` — мы ровно посередине.
*   Если `w_0 = 0.67` — мы в точке "Кот, который немного спит".

Это называется **выпуклая комбинация**. Она позволяет создать **новый смысл**, которого не было в словаре отдельно, плавно смешав существующие смыслы.



### 4. Магия "Остаточного сигнала" (Residual Connection)

В реальном Трансформере эту формулу обычно записывают так:
`Новое_слово = Старое_слово + Смесь_контекста`

То есть к исходному вектору "Пушистый" прибавляют то, что мы насмешивали из контекста.

Твой код `context_vec_2 += attn * x` делает именно это.
Мы не заменили "Пушистый" котом. Мы **обогатили** "Пушистый" знанием о том, что рядом есть кот.

Исходное слово `[0.1, 0.8, 0.3]` (признак "пушистость" горел ярко: 0.8).
После добавления контекста он получил `[0.24, 0.59, 0.48]`.
Он остался "Пушистым" (признаки никуда не делись), но слегка сдвинулся в сторону "Кота" и "Сна".



### Итог

Формула `sum(w * x)` гениальна, потому что она решает сразу три задачи:

1.  **Дифференцируемость:** Всё плавно, нейросеть может учиться.
2.  **Интерполяция:** Мы создаем новые смыслы на стыке слов («кот+спящий»), а не просто переключаемся между ними.
3.  **Обогащение:** Мы не теряем исходное слово, а добавляем к нему контекст, делая его умнее.

Если убрать умножение на веса (просто сложить), получится каша. Если убрать сложение всех векторов (оставить только один), мы потеряем нюансы. Только их комбинация дает "понимание".

---

### 3.3.2 Вычисление весов внимания для всех входных токенов

- Выше мы вычислили веса внимания и вектор контекста для ввода 2 (как показано в выделенной строке на рисунке ниже)
- Далее мы обобщаем это вычисление, чтобы вычислить все веса внимания и векторы контекста

<img src="https://camo.githubusercontent.com/1a159ccd6086e8e6218260fe23ac21024fee41640aeff038cd420645af22d2a9/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31312e77656270" width="800px">

- При самоанализе процесс начинается с вычисления показателей внимания, которые затем нормализуются для получения весов внимания, которые в сумме равны 1
- Эти веса внимания затем используются для генерации векторов контекста путем взвешенного суммирования входных данных

<img src="https://camo.githubusercontent.com/c3deab3b24155e639f46a4f118538276182ddd158a106bd49016ac7aac4edbf6/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31322e77656270" width="800px">

- Применяем предыдущий **шаг 1** ко всем попарным элементам, чтобы вычислить ненормализованную матрицу оценки внимания:

In [9]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- Мы можем добиться того же, что и вышеописанное, более эффективно с помощью матричного умножения:

In [10]:
attn_scores = inputs @ inputs.T # .T - транспонирование
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


 ---

### Зачем умножать матрицу на транспонированную матрицу в это случае?



Умножение матрицы на её транспонированную версию именно в этом случае (self-attention) преследует очень конкретную цель: **попарно сравнить каждый элемент последовательности с каждым другим (включая самого себя).**



### 1. Цель: Попарное сравнение

Представьте, что у вас есть список векторов (токенов):
*   Вектор A (первое слово)
*   Вектор B (второе слово)
*   Вектор C (третье слово)

Нам нужно заполнить вот такую таблицу (матрицу смежности):

|  \  | A | B | C |
|--||||
| **A** | A←→A | A←→B | A←→C |
| **B** | B←→A | B←→B | B←→C |
| **C** | C←→A | C←→B | C←→C |

В каждой ячейке должно стоять число — мера близости двух векторов. Стандартный способ измерить близость в векторном пространстве — **скалярное произведение (dot product)**.



### 2. Как извлечь строки и "развернуть" матрицу

Допустим, у нас `inputs` — это матрица `3×4` (3 токена, 4 признака).

```python
inputs = [
    [a1, a2, a3, a4],  # вектор A
    [b1, b2, b3, b4],  # вектор B
    [c1, c2, c3, c4]   # вектор C
]
```

**Что происходит при умножении `inputs @ inputs.T`?**

Матричное умножение устроено так: чтобы получить ячейку `[i][j]` результата, мы берем **i-ю строку левой матрицы** и умножаем на **j-й столбец правой матрицы**.

*   **Левая матрица** (`inputs`) — строки это сами векторы A, B, C.
*   **Правая матрица** (`inputs.T`) — столбцы это тоже векторы A, B, C, но благодаря транспонированию!

То есть мы перевернули матрицу на бок. Теперь при умножении мы берем строку `i` (токен) из левой матрицы и столбец `j` (тот же токен, но в виде столбца) из правой. Их произведение и дает скалярное произведение `Токен_i · Токен_j`.



### 3. Наглядный пример из мира Excel/таблиц

Если бы мы не использовали транспонирование, а попытались умножить `inputs @ inputs` (размеры `3×4 @ 3×4`), Питон выдал бы ошибку, потому что внутренние размерности не совпадают (4 ≠ 3).

Транспонирование вызывает два эффекта:
1.  **Делает умножение математически возможным:** `(3×4) @ (4×3)` → на выходе `3×3`.
2.  **Направляет расчеты в нужное русло:** Строки левой матрицы взаимодействуют со столбцами правой, которые на самом деле являются строками исходной матрицы.



### 4. Резюме: Почему именно так?

Мы делаем `inputs @ inputs.T` в self-attention, чтобы **эффективно, в одну строчку, посчитать скалярное произведение всех пар векторов**.
Результат — это матрица `N×N`, которая говорит нам: "Насколько каждый элемент последовательности похож на другой".

*Это заменяет двойной цикл `for` по `i` и `j`, делая вычисления в разы быстрее за счет оптимизаций библиотек типа NumPy/PyTorch.*

---

- Аналогично **шагу 2** ранее, мы нормализуем каждую строку так, чтобы значения в каждой строке в сумме равнялись 1:

In [11]:
# dim=-1 - применить нормализацию по последней размерности тензора attn_scores
# Если attn_scores - двумерный тензор, то он будет нормализован по столбцам так, чтобы значения в каждой строке в сумме = 1
attn_weights = torch.softmax(attn_scores, dim=-1) 
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- Быстрая проверка того, что значения в каждой строке действительно равны 1:

In [12]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Суммы 2ой строки:", row_2_sum)

print("Суммы всех строк:", attn_weights.sum(dim=-1))

Суммы 2ой строки: 1.0
Суммы всех строк: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- Применим предыдущий **шаг 3** для вычисления всех контекстных векторов:

In [13]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- В качестве проверки на работоспособность, ранее вычисленный контекстный вектор $z^{(2)} = [0.4419, 0.6515, 0.5683]$ можно найти во 2-й строке выше:

In [14]:
print("Предыдущий 2ой контекстный вектор:", context_vec_2)

Предыдущий 2ой контекстный вектор: tensor([0.4419, 0.6515, 0.5683])


## 3.4 Реализация самовнимания с обучаемыми весами

<img src="https://camo.githubusercontent.com/a8abf22af3ecfb1440541ff30c7dcd35d8b9708244d87ba62a8ad9ce8eb056cb/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31332e77656270" width="800px">

### 3.4.1 Пошаговое вычисление весовых коэффицентов внимания

- Реализация механизма саморегулирования, который используется в оригинальной архитектуре transformer, моделях GPT и большинстве других популярных LLM
- Этот механизм саморегулирования также называется "масштабируемым вниманием к точечному продукту"
- Общая идея аналогична предыдущей:
    - Мы хотим вычислить контекстные векторы в виде взвешенных сумм по входным векторам, специфичным для определенного входного элемента
    - Для этого нам нужны веса внимания
- Есть лишь незначительные отличия по сравнению с базовым механизмом концентрации внимания, представленным ранее:
    - Наиболее заметным отличием является введение весовых матриц, которые обновляются во время обучения модели
    - Эти обучаемые весовые матрицы имеют решающее значение для того, чтобы модель (в частности, модуль внимания внутри модели) могла научиться создавать "хорошие" контекстные векторы.

<img src="https://camo.githubusercontent.com/fa59be128b170ee4ff691460d592017ab8cb6e27ffad1c66bb32b899741c1ea4/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31342e77656270" width="800px">

- Шаг за шагом внедряя механизм самоконтроля, мы начнем с введения трех тренировочных весовых матриц $W_q$, $W_k$ и $W_v$.
- Эти три матрицы используются для проецирования встроенных входных токенов, $x^{(i)}$, в векторы запроса, ключа и значения посредством матричного умножения:

 - Вектор запроса: $q^{(i)} = x^{(i)}\,W_q $
 - Ключевой вектор: $k^{(i)} = x^{(i)}\,W_k $
 - Вектор значений: $v^{(i)} = x^{(i)}\,W_v $

- Размеры вложения входных данных $x$ и вектора запроса $q$ могут быть одинаковыми или разными, в зависимости от дизайна модели и конкретной реализации
- В моделях GPT входные и выходные параметры обычно одинаковы, но для иллюстрации, чтобы лучше следить за ходом вычислений, мы выбираем здесь разные входные и выходные параметры:

In [15]:
x_2 = inputs[1] # второй элемент ввода
d_in = inputs.shape[1] # размер входного вложения, d=3
d_out = 2 # размер выходного вложения, d=2

- Ниже мы инициализируем три весовые матрицы; обратите внимание, что мы устанавливаем `requires_grad=False`, чтобы уменьшить путаницу в выходных данных для иллюстрации, но если бы мы использовали весовые матрицы для обучения модели, мы бы установили `requires_grad=True` для обновления этих матриц во время обучения модели.

`requires_grad=True` — это команда для PyTorch: **«Следи за этим тензором и считай для него градиенты»**.

Две главные идеи:

1.  **Включает механизм обучения.** Когда вы делаете `forward pass` (прямой проход), PyTorch строит за кулисами граф вычислений. Для всего, что помечено `requires_grad=True`, он запоминает, как это участвовало в расчетах.

2.  **Позволяет обновлять веса.** После того как модель посчитала ошибку (loss), вызывается `.backward()`. PyTorch автоматически вычисляет градиент ошибки по всем параметрам с `requires_grad=True`. Затем оптимизатор (например, Adam) использует эти градиенты, чтобы чуть-чуть изменить значения матриц, делая модель лучше.

**Без этой строчки (`False`) тензор считается константой, и модель не будет его менять.** В примере специально матрицы специально заморожены, чтобы просто проиллюстрировать механику внимания, не обучая их.

In [16]:
torch.manual_seed(123) # фиксирует генератор случайных чисел. Теперь при каждом перезапуске скрипта torch.rand выдаст одни и те же числа (воспроизводимость экспериментов

# torch.rand(d_in, d_out) — создаёт матрицу размером d_in × d_out со случайными значениями от 0 до 1 (равномерное распределение)
# torch.nn.Parameter(..., requires_grad=False) — оборачивает эту матрицу в "параметр" (то есть говорит: это часть модели), но с requires_grad=False запрещает её обновлять при обучении (заморожена для демонстрации). Если бы было True, оптимизатор менял бы эти веса на этапе .backward()
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- Далее мы вычисляем векторы запроса, ключа и значения:

In [17]:
query_2 = x_2 @ W_query # _2, потому что это относится ко второму входному элементу
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


- Как мы можем видеть ниже, мы успешно спроецировали 6 входных токенов из 3D в 2D-пространство для встраивания:

In [18]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


- На следующем шаге, **шаг 2**, мы вычисляем ненормализованные показатели внимания, вычисляя скалярное произведение между запросом и каждым ключевым вектором:

<img src="https://camo.githubusercontent.com/00ac9bfff953cea59533a30c3705c85076e8b4b94c766eae47f6c5d7fa430ec2/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31352e77656270" width="800px">

In [19]:
keys_2 = keys[1] # Python запускает индекс с 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


- Поскольку у нас есть 6 входных данных, у нас есть 6 оценок внимания для данного вектора запроса:

In [20]:
attn_scores_2 = query_2 @ keys.T # Все оценки за внимание к данному запросу
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://camo.githubusercontent.com/5cdc90d2e404e5424625f25c9fe389ce2f95010f77a952e1d60e50831cbb60a2/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31362e77656270" width="800px">

- Далее, на шаге 3, мы вычисляем весовые коэффициенты внимания (нормализованные показатели внимания, которые в сумме равны 1), используя функцию softmax, которую мы использовали ранее
- Отличие от предыдущего заключается в том, что теперь мы оцениваем показатели внимания, деля их на квадратный корень из измерения вложения, $\sqrt{d_k}$ (т.е. `d_k**0,5`)

Почему именно $\sqrt{d_k}$ (т.е. `d_k**0,5`)?

Чтобы избежать «перегрузки» softmax, когда большие размерности векторов делают значения гигантскими, а градиенты — исчезающе малыми.

**Чуть подробнее:**
С ростом размерности $d_k$ скалярные произведения становятся всё больше. Без масштабирования их дисперсия растёт, softmax схлопывается в почти «одногорбый» вектор (близкий к one-hot), а градиенты стремятся к нулю — обучение встаёт. Деление на $\sqrt{d_k}$ удерживает дисперсию ≈ 1, и софтмакс выдаёт адекватно распределённые веса.

In [21]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://camo.githubusercontent.com/ca9da122cbe41d674059aa95eac087062f9ff9f44d0c0afe02ebe39951b3c81b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31372e77656270" width="800px">

- На **шаге 4** мы теперь вычисляем вектор контекста для входного вектора запроса 2:

In [22]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### 3.4.2 Реализация компактного класса Python для самовнимания

In [23]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


 ---

### Почему принято решение использовать три матрицы (запросов, ключей, значений)? Что это дает?



Это даёт **обучаемую проекцию входных эмбеддингов в три разных пространства**, что позволяет модели динамически перестраивать взаимодействие между токенами. Если совсем просто — чтобы модель научилась **гибко "искать", "сопоставлять" и "извлекать" информацию**.



### Аналогия для интуиции
Представьте поиск информации в базе данных:
- **Query (Запрос)** — "Что я ищу?" (вектор текущего токена, задающий критерий)
- **Key (Ключ)** — "Что это за элемент?" (векторы всех токенов, выступающие как заголовки)
- **Value (Значение)** — "Что нужно взять, если подходит?" (сами данные токена)

Три независимые проекции позволяют токену в роли *запроса* обращать внимание не на всё подряд в *значениях*, а целенаправленно сравнивать себя с *ключами*. Без них модель «смотрела бы на голое скалярное произведение входных векторов» — это слишком жёсткое ограничение.

### Главная причина: разные роли, разные проекции
Умножая вход `x` на `W_q`, `W_k` и `W_v`, мы получаем три разных представления одного и того же токена:

1.  `W_query`: Проецирует токен в роль «ищущего».
2.  `W_key`: Проецирует токен в роль «того, что ищут» (сравнивается с запросом).
3.  `W_value`: Проецирует токен в роль «содержимого, которое будет агрегироваться».

Модель на обучении подбирает эти матрицы так, чтобы запросы правильно находили подходящие ключи и извлекали из соответствующих значений нужную информацию. Без трёх матриц модель не могла бы разделить эти функции.

---

<img src="https://camo.githubusercontent.com/51962011703851c33b63877f0acd5ca56488e76f80b9b97583168473f65bd0f2/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31382e77656270" width="800px">

- Мы можем упростить описанную выше реализацию, используя линейные слои PyTorch, которые эквивалентны матричному умножению, если мы отключим блоки смещения
- Еще одно большое преимущество использования "nn.Linear" по сравнению с нашим ручным подходом "nn.Parameter(torch.rand(...)" заключается в том, что "nn.Linear" имеет предпочтительную схему инициализации веса, что приводит к более стабильному обучению модели

In [24]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


  --- 

### В чем отличие nn.Parameter() от nn.Linear()?



**Отличие `nn.Parameter()` от `nn.Linear()`:**

*   **`nn.Parameter`**: Это просто переменная-константа, которую PyTorch может оптимизировать, если указано `requires_grad=True`. Никакой встроенной логики умножения на данные тут нет, её нужно писать вручную (`x @ self.W`).
*   **`nn.Linear`**: Это готовый, полноценный слой нейросети. Внутри он уже хранит матрицу весов и вектор смещений как `nn.Parameter` и сразу делает операцию `x @ W^T + b`. С ним не нужно вручную описывать математику в `forward`. Это просто готовая и более удобная "коробка" для того же самого.



### Что такое qkv_bias?



**`qkv_bias`** — это флаг (переключатель), который управляет добавлением **смещения (bias)** к Query, Key и Value.

Простыми словами: **"Нужно ли добавлять свободный член (константу) к результату умножения?"**



### Как это работает в формулах

Без `bias` (когда `False`):
```
Query = входные_данные × W_query
```

С `bias` (когда `True`):
```
Query = входные_данные × W_query + b_query
```

То же самое для Key (`b_key`) и Value (`b_value`).



### Зачем нужно смещение?

Смещение даёт модели возможность **сдвигать** результат в любую сторону, даже если входные данные или веса близки к нулю. Это делает модель чуть гибче.

На практике в маленьких моделях это значения не имеет, но в больших трансформерах наличие/отсутствие смещения может влиять на стабильность обучения. По умолчанию в оригинальном самовнимании смещение часто **не используется** (`qkv_bias=False`).

---

- Обратите внимание, что `SelfAttention_v1` и `SelfAttention_v2` дают разные выходные данные, поскольку они используют разные начальные веса для весовых матриц

### Упражнение 3.1. Сравнение SelfAttention_v1 и SelfAttention_v2

In [25]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in, d_out = 3, 2

In [26]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

In [27]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [28]:
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

In [29]:
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

## 3.5 Сокрытие будущих слов с помощью причинно-следственного внмания

- При каузальном внимании(модель смотрит в прошлое, не заглядываю в будущее) веса внимания над диагональю маскируются, гарантируя, что для любого заданного входного сигнала LLM не сможет использовать будущие токены при вычислении контекстных векторов с весом внимания.

<img src="https://camo.githubusercontent.com/fb6c87d07ca30ca5a75d0682c0151d7ac3597a1d613f96f416cde10782cc78cf/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31392e77656270" width="800px">

 ---

### Зачем мы скрываем будущие слова с помощью причинно-следственного внимания?

Мы скрываем будущие слова, чтобы модель **училась предсказывать следующее слово, а не списывать его**.

Если бы модель видела всё предложение сразу, она бы просто "подглядывала" правильный ответ вместо того, чтобы учиться генерировать текст по порядку.

### Аналогия:
Представьте, что вы учите ребёнка читать и просите продолжить фразу: *"В лесу родилась..."*
- Если перед ним уже лежит полный текст, он просто прочитает слово *"ёлочка"* — никакого обучения не произошло.
- Если он видит только начало, ему приходится **думать и предсказывать** — и только так он учится.

То же самое с языковыми моделями: казуальное внимание заставляет их **генерировать текст шаг за шагом, опираясь только на контекст слева**, что и происходит при реальном использовании (мы же не знаем, что модель скажет через слово).

---

### 3.5.1 Применение маски причинно-следственного внимания

- Мы преобразуем предыдущий механизм самонаблюдения в механизм каузального самонаблюдения
- Каузальное самонаблюдение гарантирует, что предсказание модели для определенной позиции в последовательности зависит только от известных результатов в предыдущих позициях, а не от будущих позиций
- Проще говоря, это гарантирует, что предсказание каждого следующего слова должно зависеть только от предыдущих слов
- Для достижения этой цели для каждого заданного токена мы маскируем будущие токены (те, которые следуют за текущим токеном во входном тексте):

<img src="https://camo.githubusercontent.com/d667abd2d47a2fb792d2f59e6756518e6b6a09f4b2c5e515df227d23af56b415/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32302e77656270" width="800px">

In [30]:
# Переиспользование матрицы запросов, ключевых весов объекта и SelfAttention_v2
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


- Самый простой способ замаскировать будущие значения концентрации внимания - это создать маску с помощью функции tril от PyTorch, установив для элементов ниже главной диагонали (включая саму диагональ) значение 1, а для элементов над главной диагональю значение 0:

In [31]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length)) # torch.tril() берет матрицу и обнуляет всё, что выше главной диагонали, оставляя только нижнюю треугольную часть
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- Затем мы можем умножить показатели внимания с помощью этой маски, чтобы обнулить показатели внимания выше диагонали:

In [32]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


- Однако, если бы маска была применена после softmax, как указано выше, это нарушило бы распределение вероятностей, созданное softmax
- Softmax гарантирует, что все выходные значения в сумме равны 1
- Маскировка после softmax потребует повторной нормализации выходных данных для суммирования до 1, что усложняет процесс и может привести к непредвиденным последствиям

- Чтобы убедиться, что сумма строк равна 1, мы можем нормализовать весовые коэффициенты внимания следующим образом:

In [33]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


 ---

### Что делает код выше?

Этот код **нормализует каждую строку матрицы так, чтобы сумма значений в строке стала равна 1**.

### По шагам:

**1. `masked_simple.sum(dim=-1, keepdim=True)`**
Суммирует все числа в каждой строке. `dim=-1` — последнее измерение (столбцы). `keepdim=True` сохраняет размерность, чтобы потом можно было делить.

**2. `masked_simple / row_sums`**
Делит каждое значение на сумму его строки.

### Простой пример:

**Исходная матрица `masked_simple`:**
```
1 0 0
1 1 0
1 1 1
```

**Суммы строк (`row_sums`):**
```
1
2
3
```

**После деления (результат):**
```
1.00  0.00  0.00
0.50  0.50  0.00
0.33  0.33  0.33
```

Теперь каждая строка в сумме даёт 1 — это веса внимания, которые говорят: "на какой токен сколько процентов внимания обратить".

---

- Хотя технически мы уже закончили с кодированием механизма каузального внимания, давайте вкратце рассмотрим более эффективный подход для достижения того же, что и вышеописанный
- Таким образом, вместо того, чтобы обнулять значения концентрации внимания выше диагонали и перенормировать результаты, мы можем замаскировать ненормализованные значения концентрации внимания выше диагонали отрицательной бесконечностью, прежде чем они попадут в функцию softmax:

<img src="https://camo.githubusercontent.com/86020330d6b90e8e761e1f2b7caeb5c5687c69f3465771466ea3356a91bf5929/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32312e77656270" width="800px">

In [34]:
# torch.ones() - cоздаёт квадратную матрицу из единиц
# torch.triu(..., diagonal=1) - triu = "triangle upper" = верхний треугольник. Оставляет единицы выше диагонали (начиная с diagonal=1, то есть не включая саму диагональ), всё остальное обнуляет
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

# В матрице attn_scores заменяет все позиции, где в маске стоит 1 (True), на минус бесконечность. Позиции с нулём остаются без изменений
masked = attn_scores.masked_fill(mask.bool(), -torch.inf) 
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


 ---

### Почему этот подход эффективнее?

Потому что **softmax игнорирует отрицательную бесконечность** (`exp(-∞) = 0`), и строчки автоматически нормализуются в сумму = 1 без ручного пересчёта, который может испортить градиенты и числовую стабильность.

**Проще говоря:**
- **Старый способ**: обнулили → разрушили нормировку → пришлось вручную пересчитывать (два действия).
- **Новый способ**: заменили на `-inf` → softmax сам превратил это в 0 и сразу выдал правильные вероятности в сумме 1 (одно действие, без ручной возни).

---

- Как мы можем видеть ниже, теперь значения коэффициента внимания в каждой строке снова корректно суммируются с 1:

In [35]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 Маскирование дополнительных весов внимания с помощью отсева

- Кроме того, мы также применяем отсев, чтобы уменьшить переобучение во время обучения
- Отсев может быть применен в нескольких местах:
    - например, после вычисления весов внимания;
    - или после умножения весов внимания на векторы значений
- Здесь мы применим маску отсева после вычисления весов внимания, потому что это более распространено

- Кроме того, в этом конкретном примере мы используем коэффициент отсева в 50%, что означает случайное маскирование половины весов внимания. Когда мы позже будем обучать модель GPT, мы будем использовать более низкий коэффициент отсева, например, 0,1 или 0,2

<img src="https://camo.githubusercontent.com/93bf2809e9516322e422afdc77a213a4ae4549089d2aa64ecac34ea37ea99de1/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32322e77656270" width="800px">

- Если мы применим коэффициент отсева, равный 0,5 (50%), то значения, которые не были отсеяны, будут соответствующим образом увеличены с коэффициентом 1/0,5 = 2
- Масштабирование рассчитывается по формуле 1 / (1 - "коэффициент отсева").

In [36]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # процент отсева составляет 50%
example = torch.ones(6, 6) # создание матрицы из единиц

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [37]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


- Обратите внимание, что результирующие выходные данные о выпадении могут выглядеть по-разному в зависимости от вашей операционной системы; вы можете прочитать больше об этом несоответствии [здесь, на странице отслеживания проблем с PyTorch](https://github.com/pytorch/pytorch/issues/121595)

 ---

### Почему отсев реализован именно так?

**Дропаут (Dropout)** — это метод регуляризации нейросетей, который случайным образом "выключает" часть нейронов во время обучения, чтобы модель не переобучалась и не полагалась на конкретные связи.

Ваш код `torch.nn.Dropout(0.5)` создаёт слой, который с вероятностью **50% обнуляет** каждый входной элемент и **увеличивает оставшиеся значения в 2 раза**, чтобы сохранить среднюю сумму активаций неизменной.

### Как это устроено

1. **Исходная матрица `example`**: 6×6, заполнена единицами.
2. **Маска дропаута**: генерируется случайная булева матрица 6×6, где каждое значение имеет 50% шанс быть `True` (оставить нейрон) или `False` (выключить нейрон).
3. **Обнуление**: элементы, соответствующие `False`, заменяются на `0`.
4. **Масштабирование (inverted dropout)**: оставшиеся значения умножаются на `1 / (1 - 0.5) = 2`. Это нужно, чтобы ожидаемая сумма значений по всему слою не менялась — иначе следующий слой видел бы в среднем вдвое меньший сигнал во время обучения по сравнению с инференсом, когда дропаут отключается.

### Что показывает `print(dropout(example))`

Вы увидите матрицу, где примерно половина элементов — `0`, а вторая половина — `2` (исходная единица × 2). При этом на этапе **инференса (предсказания)** дропаут автоматически отключается, и значения проходят без изменений, но модель уже обучена быть устойчивой к отсутствию любых отдельных связей.

---

### 3.5.3 Реализация компактного класса причинно-следственного внимания

- Теперь мы готовы внедрить рабочую реализацию самоконтроля, включая маски причинности и отсева
- Еще одна задача - реализовать код для обработки пакетов, состоящих из более чем одного ввода, чтобы наш класс `CausalAttention` поддерживал пакетные выходные данные, создаваемые загрузчиком данных, который мы реализовали в главе 2
- Для простоты, чтобы имитировать такой пакетный ввод, мы дублируем пример ввода текста:

In [38]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 ввода данных по 6 токенов в каждом, и каждый токен имеет размер встраивания 3

torch.Size([2, 6, 3])


 ---


###  Наглядный пример

```python
# Исходные данные: 6 токенов, размерность эмбеддинга = 3
inputs = torch.tensor([
    [0.1, 0.2, 0.3],  # токен 1
    [0.4, 0.5, 0.6],  # токен 2
    [0.7, 0.8, 0.9],  # токен 3
    [1.0, 1.1, 1.2],  # токен 4
    [1.3, 1.4, 1.5],  # токен 5
    [1.6, 1.7, 1.8]   # токен 6
])

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  # torch.Size([2, 6, 3])
print(batch)
```

**Вывод:**
```
tensor([
    [[0.1000, 0.2000, 0.3000],
     [0.4000, 0.5000, 0.6000],
     [0.7000, 0.8000, 0.9000],
     [1.0000, 1.1000, 1.2000],
     [1.3000, 1.4000, 1.5000],
     [1.6000, 1.7000, 1.8000]],

    [[0.1000, 0.2000, 0.3000],
     [0.4000, 0.5000, 0.6000],
     [0.7000, 0.8000, 0.9000],
     [1.0000, 1.1000, 1.2000],
     [1.3000, 1.4000, 1.5000],
     [1.6000, 1.7000, 1.8000]]
])
```

---

In [39]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # Новое
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # Новое

    def forward(self, x):
        b, num_tokens, d_in = x.shape # Новое пакетное измерение b
        # Для входных данных, где `num_tokens` превышает `context_length`, это приведет к ошибкам
        # при создании маски, приведенной ниже.
        # На практике это не проблема, поскольку LLM гарантирует, что входные данные 
        # не превышают `context_length` до перехода к этому прямому методу.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Измененное транспонирование
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` для учета случаев, когда количество токенов в пакете меньше поддерживаемого context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


 ---

### Что делает код выше?

Код реализует **causal attention (причинное внимание)** для пакетной обработки данных.

### Архитектура:

**`__init__`:**
- `W_query`, `W_key`, `W_value` — три линейных слоя (обучаемые матрицы) для проекции в Query, Key, Value
- `dropout` — слой регуляризации
- `register_buffer('mask')` — создаёт верхнетреугольную маску (`triu`), которая хранится как буфер (не обучается, но сохраняется вместе с моделью)

**`forward`:**
1. **Распаковка батча** `b, num_tokens, d_in = x.shape`:
   - `b` — количество примеров в пакете (batch size)
   - `num_tokens` — число токенов
   - `d_in` — размерность входного эмбеддинга

2. **Проекция** входов через `W_query`, `W_key`, `W_value`

3. **Вычисление attention scores**:
   ```python
   attn_scores = queries @ keys.transpose(1, 2)
   ```
   - `keys.transpose(1, 2)` — транспонирует два последних измерения для правильного умножения в батче
   - Результат: `(b, num_tokens, num_tokens)`

4. **Маскирование будущих токенов**:
   ```python
   attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
   ```
   - Заменяет позиции выше диагонали на `-inf`
   - `[:num_tokens, :num_tokens]` — адаптация маски, если токенов меньше `context_length`

5. **Softmax + масштабирование** на `sqrt(d_k)`

6. **Dropout** на весах внимания

7. **Взвешенная сумма** значений: `attn_weights @ values`

### Результат:
`context_vecs.shape` → `(2, 6, 3)` — батч из 2 примеров, 6 токенов, каждый размерности `d_out=3`.

---

- Обратите внимание, что отсев применяется только во время обучения, а не во время вывода

<img src="https://camo.githubusercontent.com/7b52492f7e7687e2931f7d33a30e6d1f3d7f8e125012a678dfa69a1709ed0e34/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32332e77656270" width="800px">

## 3.6 Расширение одноцелевого внимания до многоцелевого

### 3.6.1 Объединение нескольких одноцелевых слоев внимания

- Ниже приведена краткая информация о ранее реализованном самонаблюдении (причинно-следственные связи и маски выпадения не показаны для простоты)

- Это также называется вниманием с одной целью:

<img src="https://camo.githubusercontent.com/a77ef946eb06252af4405c7321883c90a254f4e1647680a754e9c628fae3c369/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32342e77656270" width="800px">

- Мы просто складываем несколько модулей внимания с одной целью, чтобы получить модуль внимания с несколькими целями:

<img src="https://camo.githubusercontent.com/a17e377668bf3788856f243e321da70c0fd8d792b6dd0af089a18fbbdc00579b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32352e77656270" width="800px">

- Основная идея многозадачного управления вниманием заключается в многократном запуске механизма управления вниманием (параллельно) с различными изученными линейными проекциями. Это позволяет модели совместно обрабатывать информацию из разных подпространств представления в разных позициях.

In [40]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # Это количество токенов
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


- В приведенной выше реализации размерность встраивания равна 4, потому что мы используем `d_out=2` в качестве размера встраивания для векторов ключа, запроса и значения, а также вектора контекста. И поскольку у нас есть 2 цели внимания, мы получаем размерность вложения выходных данных 2*2=4

 ---

### Что делает код выше?

**1. `__init__`:**
```python
self.heads = nn.ModuleList([
    CausalAttention(...) for _ in range(num_heads)
])
```
- Создаёт список из `num_heads` независимых целей causal attention
- Каждая цель имеет свои матрицы `W_query`, `W_key`, `W_value`

**2. `forward`:**
```python
return torch.cat([head(x) for head in self.heads], dim=-1)
```
- Пропускает на вход `x` через каждую цель независимо
- Конкатенирует результаты по последнему измерению (`dim=-1`)

### Размерности:

- Вход: `(2, 6, 3)` — батч, 6 токенов, `d_in=3`
- Одна цель выдаёт: `(2, 6, 2)` — `d_out=2`
- Две цели → конкатенация: `(2, 6, 4)` — `d_out * num_heads = 2 * 2 = 4`

### Результат:
```
context_vecs.shape: torch.Size([2, 6, 4])
```

Каждый токен теперь представлен контекстным вектором размерности 4, где первые 2 числа — от первой цели, вторые 2 числа — от второй. Цели учатся разным паттернам внимания (одна может фокусироваться на синтаксисе, другая на семантике).

---

### Упражнение 3.2. Возврат двумерных векторов вложения

Если мы хотим получить размерность вывода, равную 2, как ранее в примере с одной целью, нам, возможно, придется изменить размерность проекции `d_out` на 1:

In [41]:
torch.manual_seed(123)

d_out = 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### 3.6.2 Реализация многоцелевого внимания с разделением весов

- Хотя вышеописанное является интуитивно понятной и полнофункциональной реализацией multihead attention (завершающей реализацию CausalAttention, описанную ранее), мы можем написать отдельный класс под названием "MultiHeadAttention" для достижения того же результата.

- Мы не объединяем отдельные заголовки внимания для этого отдельного класса "MultiHeadAttention"
- Вместо этого мы создаем отдельные весовые матрицы W_query, W_key и W_value, а затем разбиваем их на отдельные матрицы для каждого заголовка внимания:

In [42]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Уменьшите dim проекции, чтобы она соответствовала желаемой dim выходного сигнала.

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Линейный слой для объединения выходных данных головки
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # Как и в `CausalAttention`, для входных данных, где `num_tokens` превышает `context_length`,
        # это приведет к ошибкам при создании маски, описанным ниже. 
        # На практике это не проблема, поскольку LLM гарантирует, что входные данные 
        # не превышают `context_length` до перехода к этому прямому методу.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Мы неявно разделяем матрицу, добавляя измерение "num_heads"
        # Разворачиваем последний dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Транспонировать: (b, num_tokens, num_heads, head_dim) - > (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Вычислите масштабированное внимание к точечному продукту (оно же внимание к себе) с помощью причинно-следственной маски
        attn_scores = queries @ keys.transpose(2, 3)  # Точечный продукт для каждой цели

        # Исходная маска усекается до количества токенов и преобразуется в логическое значение
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Используйте маску, чтобы набирать баллы за внимание
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Форма: (b, num_tokens, num_heads, head_dim)   
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Комбинат глав, где самообслуживание.d_out = самоуправления.num_heads * самовывозом.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- Обратите внимание, что приведенное выше - это, по сути, переписанная версия "MultiHeadAttentionWrapper", которая является более эффективной
- Результирующий набор выглядит немного по-другому, поскольку инициализации случайного веса различаются, но обе они являются полнофункциональными реализациями, которые можно использовать в классе GPT

---

** Примечание о выходных параметрах**

- В приведенном выше примере "MultiHeadAttention" я использовал "d_out=2", чтобы использовать ту же настройку, что и в классе "MultiHeadAttentionWrapper" ранее
- `MultiHeadAttentionWrapper`, из-за объединения, возвращает выходной размер цели `d_out * num_heads` (т.е. `2*2 = 4`).
- Однако класс `MultiHeadAttention` (чтобы сделать его более удобным для пользователя) позволяет нам управлять выходным размером цели напрямую через `d_out`; это означает, что если мы установим `d_out = 2`, выходной размер цели будет равен 2, независимо от количества целей
- Оглядываясь назад, как [отметили] читатели(https://github.com/rasbt/LLMs-from-scratch/pull/859 ), возможно, было бы более интуитивно использовать `MultiHeadAttention` с `d_out = 4`, чтобы он выдавал те же выходные данные, что и `MultiHeadAttentionWrapper` с `d_out = 2`.

---

- Обратите внимание, что кроме того, мы добавили слой линейной проекции (`self.out_proj `) в класс `MultiHeadAttention`, указанный выше. Это просто линейное преобразование, которое не изменяет размеры. Использование такого проекционного слоя в реализации LLM является стандартным соглашением, но это не является строго необходимым (недавние исследования показали, что его можно удалить, не влияя на производительность моделирования).

<img src="https://camo.githubusercontent.com/f15b686e20a63ad474d1a8b0cd6d4c7550565b9e73c5e33207303907ea903fcd/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32362e77656270" width="800px">

- Обратите внимание, что если вы заинтересованы в компактной и эффективной реализации вышеописанного, вы также можете рассмотреть класс [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) в PyTorch

- Поскольку приведенная выше реализация на первый взгляд может показаться немного сложной, давайте посмотрим, что происходит при выполнении `attn_scores = queries @ keys.transpose(2, 3)`:

In [43]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- В этом случае реализация матричного умножения в PyTorch будет обрабатывать 4-мерный входной тензор таким образом, что матричное умножение выполняется между двумя последними измерениями (num_tokens, head_dim), а затем повторяется для отдельных целей 

- Например, следующий способ становится более компактным для вычисления матричного умножения для каждой цели в отдельности:

In [44]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("Первая цель:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nВторая цель:\n", second_res)

Первая цель:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Вторая цель:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


### Упражнение 3.3. Инициализация модулей внимания размером с GPT-2

In [45]:
context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)

При желании можно указать следующее количество параметров:

In [46]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(mha)

2360064

Модель GPT-2 имеет в общей сложности 117 миллионов параметров, но, как мы видим, большинство из них находятся не в самом модуле multi-head attention.